# Class Exercises 6

**Basic Algorithms I**

> **Jupyter version — same exercise set**
> Standalone notebook; no download or external checker.py is needed.

All seven tasks are class exercises. Suggested time: 10 minutes per task, and 15 minutes for Exercise 7 (75 minutes total).

**Programming rules:** use the named algorithms with explicit loops and conditions. Do not use comprehensions, `break`, `continue`, `return`, or shortcut functions such as `sum()`, `max()`, `any()`, `index()`, or `count()`. Use `random` only in Exercises 2, 4, 6 and 7. Meaningful input prompts and output labels are allowed. Indexes start at 0; Exercise 4 asks for a toss number starting at 1.

Checks are learning feedback, not graded assessment or proof that every algorithmic detail is correct. Your normal solution cell runs in the notebook kernel; the checker runs its test cases in separate processes with a three-second timeout.

In [ ]:
# Setup – run this cell once before starting.
# Standalone Week 6 checker; no download or external checker.py is needed.
import ast, json, re, subprocess, sys
from IPython import get_ipython

_STRUCTURE_SOURCE = '\nimport ast,json\ndef _st(src):\n t=ast.parse(src)\n calls=[n for n in ast.walk(t) if isinstance(n,ast.Call)]\n imports=[]\n for n in ast.walk(t):\n  if isinstance(n,ast.Import): imports += [a.name for a in n.names]\n  elif isinstance(n,ast.ImportFrom) and n.module: imports.append(n.module)\n fors=[n for n in ast.walk(t) if isinstance(n,ast.For)]\n whiles=[n for n in ast.walk(t) if isinstance(n,ast.While)]\n forbidden=[type(n).__name__.lower() for n in ast.walk(t) if isinstance(n,(ast.Break,ast.Continue,ast.Return))]\n comprehensions=any(isinstance(n,(ast.ListComp,ast.SetComp,ast.DictComp,ast.GeneratorExp)) for n in ast.walk(t))\n shortcut_names={"sum","min","max","any","all","sorted","enumerate","zip","next","eval","exec","getattr","__import__","map","filter","reversed"}\n shortcuts=[n.func.id for n in calls if isinstance(n.func,ast.Name) and n.func.id in shortcut_names]\n methods=[n.func.attr for n in calls if isinstance(n.func,ast.Attribute) and not (isinstance(n.func.value,ast.Name) and n.func.value.id=="random")]\n while_has_and=any(isinstance(n.test,ast.BoolOp) and isinstance(n.test.op,ast.And) for n in whiles)\n return json.dumps({\n "input":any(isinstance(n,ast.Call) and isinstance(n.func,ast.Name) and n.func.id=="input" for n in ast.walk(t)),\n "if":any(isinstance(n,ast.If) for n in ast.walk(t)),\n "comparison":any(isinstance(n,ast.Compare) for n in ast.walk(t)),\n "loop":bool(fors or whiles),\n "for":bool(fors),"while":bool(whiles),"while_has_and":while_has_and,\n "print":any(isinstance(n,ast.Call) and isinstance(n.func,ast.Name) and n.func.id=="print" for n in ast.walk(t)),\n "random":any(name=="random" for name in imports),\n "imports":imports,"forbidden":forbidden,"comprehensions":comprehensions,\n "shortcuts":shortcuts,"methods":methods,\n "membership":any(isinstance(op,(ast.In,ast.NotIn)) for n in ast.walk(t) if isinstance(n,ast.Compare) for op in n.ops),\n "collections":{\n   "list":any(isinstance(n,ast.List) for n in ast.walk(t)),\n   "tuple":any(isinstance(n,ast.Tuple) for n in ast.walk(t)),\n   "set":any(isinstance(n,ast.Set) for n in ast.walk(t)),\n   "dict":any(isinstance(n,ast.Dict) for n in ast.walk(t)),\n   "constructors":[n.func.id for n in calls if isinstance(n.func,ast.Name) and n.func.id in {"list","tuple","set","dict"}]\n }\n })\n_st(student_code)\n'
_MATCH_SOURCE = '\nimport ast,re,json,math\nvalues=[]\nlines=[line.strip() for line in check_output.splitlines() if line.strip()]\nfor line in lines:\n    for start in [m.start() for m in re.finditer(r"[\\[\\{\\(]",line)]:\n        try:\n            value=ast.literal_eval(line[start:].strip())\n            if isinstance(value,(list,tuple,dict,set)):\n                values.append(value)\n                break\n        except (ValueError,SyntaxError):\n            pass\ndef nums(line):\n    return [float(v) for v in re.findall(r"[-+]?(?:\\d+(?:\\.\\d*)?|\\.\\d+)(?:[eE][-+]?\\d+)?",line)]\ndef last_number():\n    v=nums(lines[-1]) if lines else []\n    return v[-1] if v else None\ninputs=json.loads(check_inputs)\nvalid=False\nif check_id=="ex01":\n    expected=any(float(v)>185 for v in inputs)\n    tail=lines[-1].lower() if lines else ""\n    words=re.findall(r"\\b(true|false)\\b",tail)\n    valid=bool(words) and (words[-1]=="true")==expected\nelif check_id=="ex02":\n    candidates=[v for v in values if isinstance(v,tuple) and len(v)==7]\n    if candidates:\n        v=candidates[-1]\n        valid=all(type(t) in (int,float) and math.isfinite(t) and 10<=t<25 for t in v) and last_number() is not None and abs(last_number()-sum(v)/7)<=0.011\nelif check_id=="ex03":\n    n=int(inputs[0]);count=sum(n%d==0 for d in range(1,n+1))\n    tail=lines[-1].lower() if lines else ""\n    is_prime=bool(re.search(r"\\bprime\\b",tail)) and not re.search(r"\\bnot\\b|\\bnon[- ]?prime\\b|\\bcomposite\\b",tail)\n    reported=nums(lines[-2]) if len(lines)>=2 else []\n    valid=bool(reported) and reported[-1]==count and bool(re.search(r"\\bprime\\b|\\bcomposite\\b",tail)) and is_prime==(count==2)\nelif check_id=="ex04":\n    candidates=[v for v in values if isinstance(v,list) and 10<=len(v)<=100]\n    if candidates:\n        v=candidates[-1]\n        valid=all(t in ("heads","tails") for t in v) and "heads" in v and last_number()==v.index("heads")+1\nelif check_id=="ex05":\n    names=inputs[1:];expected=names.index("Peter") if "Peter" in names else -1\n    valid=last_number()==expected\nelif check_id=="ex06":\n    candidates=[v for v in values if isinstance(v,list) and len(v)==15]\n    if candidates:\n        v=candidates[-1]\n        valid=all(type(t) is int and 1000<=t<=2023 for t in v) and last_number()==max(v)\nelif check_id=="ex07":\n    bounds=None\n    for line in lines[:-1]:\n        v=nums(line)\n        if len(v)>=2:\n            lo,hi=v[-2:]\n            if lo.is_integer() and hi.is_integer() and 1<=lo<=50 and lo<=hi<=1000:\n                bounds=(int(lo),int(hi));break\n    if bounds:\n        lo,hi=bounds;expected=next((n for n in (6,28,496) if lo<=n<=hi),None)\n        tail=lines[-1].lower()\n        no_match=bool(re.search(r"\\bno\\b|\\bnot found\\b|\\bnone\\b",tail))\n        valid=(no_match and "perfect" in tail) if expected is None else (not no_match and last_number()==expected)\nvalid\n'
_EXERCISES = {'ex01': {'tests': [['170', '180', '165', '185', '175', '182', '179', '168', '184', '181'], ['190', '170', '180', '165', '185', '175', '182', '179', '168', '184'], ['170', '180', '165', '185', '175', '182', '179', '168', '184', '186']], 'need': ['input', 'loop', 'while', 'while_has_and', 'print'], 'starter': 'heights = []\n# Read the 10 heights and append them to the list.\n# Then apply the decision algorithm.\n'}, 'ex02': {'tests': [[]], 'need': ['random', 'loop', 'print'], 'starter': 'import random\n\ntemperatures = []\n# Generate seven values, then convert the list to a tuple.\n# Compute their average using an accumulator.\n'}, 'ex03': {'tests': [['1'], ['2'], ['12'], ['17']], 'need': ['input', 'loop', 'if', 'print'], 'starter': 'n = int(input("Enter a positive integer: "))\n# Count divisors, then decide whether n is prime.\n'}, 'ex04': {'tests': [[]], 'need': ['random', 'loop', 'while', 'print'], 'starter': 'import random\n\nn = random.randint(10, 100)\ntosses = []\n# Generate a sequence containing at least one heads.\n# Apply the selection algorithm to find the first heads.\n'}, 'ex05': {'tests': [['3', 'Anna', 'Peter', 'Ben'], ['2', 'Peter', 'Peter'], ['3', 'Anna', 'Ben', 'Peter'], ['3', 'Anna', 'peter', 'Ben'], ['1', 'Peter']], 'need': ['input', 'loop', 'while', 'while_has_and', 'print'], 'starter': 'n = int(input("Number of names: "))\nnames = []\n# Read the names, then apply linear search.\n'}, 'ex06': {'tests': [[]], 'need': ['random', 'loop', 'if', 'print'], 'starter': 'import random\n\nnumbers = []\n# Generate 15 values and apply maximum selection.\n'}, 'ex07': {'tests': [[]], 'need': ['random', 'loop', 'while', 'while_has_and', 'if', 'print'], 'starter': 'import random\n\nlower = random.randint(1, 50)\nupper = random.randint(lower, 1000)\nprint("Interval:", lower, upper)\n# Search for the first perfect number in the interval.\n'}}

_RUNNER = r"""
import sys,json,io,builtins,contextlib,random
payload=json.loads(sys.stdin.read())
values=iter(payload["inputs"])
class LimitedOutput(io.StringIO):
    def write(self,text):
        if self.tell()+len(text)>50000: raise RuntimeError("Output limit exceeded.")
        return super().write(text)
output=LimitedOutput()
def inp(prompt=""):
    try: return str(next(values))
    except StopIteration: raise EOFError("More input values requested than expected.")
builtins.input=inp
random.seed(517)
namespace={"__name__":"__main__"}
try:
    with contextlib.redirect_stdout(output),contextlib.redirect_stderr(output):
        exec(compile(payload["code"],"<student_code>","exec"),namespace,namespace)
    result={"ok":True,"output":output.getvalue()}
except Exception as error:
    result={"ok":False,"output":output.getvalue(),"error":type(error).__name__+": "+str(error)}
sys.stdout.write(json.dumps(result))
"""

def _exercise_id(task_id):
    if not re.fullmatch(r"week06_ex0[1-7]",task_id):
        raise ValueError("Unknown Week 6 task: "+str(task_id))
    return task_id.split("_")[1]

def _previous_solution():
    shell=get_ipython()
    if shell is None: raise RuntimeError("Run this checker inside a notebook.")
    for source in reversed(shell.history_manager.input_hist_raw[1:]):
        if not source.strip(): continue
        if "_STRUCTURE_SOURCE" in source: continue
        try: tree=ast.parse(source)
        except SyntaxError: return source
        checking=any(isinstance(n,ast.Call) and isinstance(n.func,ast.Name) and
                     n.func.id in {"check","check_button","check_solution"} for n in ast.walk(tree))
        if not checking: return source
    raise RuntimeError("Run your solution cell first, then run its check cell.")

def _rule_problems(exercise, st):
    problems=[]
    if st["forbidden"]: problems.append("Do not use break, continue or return as loop-control shortcuts.")
    if st["comprehensions"]: problems.append("Use explicit loops rather than comprehensions.")
    if st["shortcuts"]: problems.append("Avoid shortcut functions: "+", ".join(st["shortcuts"]))
    allowed=["random"] if exercise in {"ex02","ex04","ex06","ex07"} else []
    if any(module not in allowed for module in st["imports"]): problems.append("Only random is permitted, and only in Exercises 2, 4, 6 and 7.")
    if any(m in {"index","count","sort","find"} for m in st["methods"]): problems.append("Implement the required algorithm explicitly; do not use index(), count(), sort() or find().")
    if exercise=="ex05" and st.get("membership"): problems.append("Use element-by-element linear search, not a membership test for Peter.")
    if exercise=="ex02" and not (st["collections"]["tuple"] or "tuple" in st["collections"]["constructors"]): problems.append("Store the seven temperatures in a tuple.")
    return problems

def check_solution(task_id,code):
    exercise=_exercise_id(task_id);info=_EXERCISES[exercise]
    try:
        namespace={"student_code":code}
        exec(_STRUCTURE_SOURCE,namespace)
        st=json.loads(namespace["_st"](code))
    except SyntaxError as error:
        print(f"✗ Syntax error on line {error.lineno}: {error.msg}");return False
    problems=_rule_problems(exercise,st)
    missing=[concept for concept in info["need"] if not st.get(concept)]
    if missing: problems.append("Expected structure: "+", ".join(missing))
    if problems:
        print("✗ Programming rules / expected structure:")
        for problem in problems: print("-",problem)
        return False
    passed=0
    for inputs in info["tests"]:
        try:
            result=subprocess.run([sys.executable,"-c",_RUNNER],
                input=json.dumps({"code":code,"inputs":inputs}),text=True,
                capture_output=True,timeout=3)
        except subprocess.TimeoutExpired:
            print("✗ Execution timed out (3 seconds). Check your loop conditions.");return False
        if result.returncode:
            print("✗ Python process failed:",result.stderr[:1000]);return False
        try: run=json.loads(result.stdout)
        except (ValueError,TypeError):
            print("✗ The checker could not read the execution result.");return False
        if not run["ok"]:
            print("✗ Runtime error:",run["error"]);return False
        namespace={"check_id":exercise,"check_output":run["output"],"check_inputs":json.dumps(inputs)}
        exec(_MATCH_SOURCE,namespace)
        if namespace["valid"]: passed+=1
    ok=passed==len(info["tests"])
    print(("✓ Correct! " if ok else "✗ Review your result. ")+f"Functional tests: {passed}/{len(info['tests'])}.")
    if not ok:
        print("Use the requested result order. Labels and input prompts are allowed. Check index versus toss number and interval bounds.")
    return ok

def check(task_id):
    return check_solution(task_id,_previous_solution())

def check_button(task_id,label="Check my solution"):
    import ipywidgets as widgets
    from IPython.display import display
    # Capture this exercise's executed solution, so other check buttons do not change its target.
    source=_previous_solution()
    button=widgets.Button(description=label,button_style="primary")
    output=widgets.Output()
    def clicked(_):
        with output:
            output.clear_output()
            check_solution(task_id,source)
    button.on_click(clicked)
    display(button,output)

print("✓ Week 6 checker is ready.")


## Exercise 1 – Heights – decision (10 minutes)

There are 10 students in a class. Read their heights in centimetres and store them in a list. Use the decision algorithm to determine whether at least one student is taller than 185 cm. Print a labelled True/False result. Stop the search as soon as a match is found; 185 cm itself does not qualify.

<details>
<summary><strong>Algorithm / pseudocode</strong> (click to expand)</summary>

```text
heights ← empty list
REPEAT 10 TIMES:
    READ height
    APPEND height TO heights
i ← 0
WHILE i < LENGTH(heights) AND heights[i] <= 185:
    i ← i + 1
exists ← (i < LENGTH(heights))
DISPLAY a label and exists
```

</details>


In [ ]:
heights = []
# Read the 10 heights and append them to the list.
# Then apply the decision algorithm.


**Check your solution**

Run this cell immediately after running your solution.


In [ ]:
check("week06_ex01")


## Exercise 2 – Weekly temperature – summarization (10 minutes)

Generate the average temperature for each of the seven days of a week as a random floating-point number in the interval [10.0, 25.0). Store the seven values in a tuple. Display the tuple and the weekly average. Use the summarization algorithm with an explicit loop, not sum(). For generation, random.random() * 15 + 10 gives the required interval; you may build a list first, then convert it with tuple(). Print the average on the final output line, optionally with a label; rounding to two decimal places is sufficient.

<details>
<summary><strong>Algorithm / pseudocode</strong> (click to expand)</summary>

```text
values ← empty list
REPEAT 7 TIMES:
    APPEND random real number from [10.0, 25.0) TO values
temperatures ← TUPLE(values)
total ← 0
FOR each temperature IN temperatures:
    total ← total + temperature
average ← total / 7
DISPLAY temperatures
DISPLAY a label and average
```

</details>


In [ ]:
import random

temperatures = []
# Generate seven values, then convert the list to a tuple.
# Compute their average using an accumulator.


**Check your solution**

Run this cell immediately after running your solution.


In [ ]:
check("week06_ex02")


## Exercise 3 – Divisors and prime numbers – counting (10 minutes)

Ask for a positive integer n. Use the counting algorithm to count its positive divisors, including 1 and n. Display the divisor count on one line and a labelled prime/not-prime message on the final line (for example, "This is a prime number." or "This is not a prime number."). A number is prime exactly when it has two positive divisors; 1 is not prime.

<details>
<summary><strong>Algorithm / pseudocode</strong> (click to expand)</summary>

```text
READ n
count ← 0
FOR divisor FROM 1 TO n:
    IF n MOD divisor = 0:
        count ← count + 1
DISPLAY a label and count
IF count = 2:
    DISPLAY "This is a prime number."
ELSE:
    DISPLAY "This is not a prime number."
```

</details>


In [ ]:
n = int(input("Enter a positive integer: "))
# Count divisors, then decide whether n is prime.


**Check your solution**

Run this cell immediately after running your solution.


In [ ]:
check("week06_ex03")


## Exercise 4 – First heads – selection (10 minutes)

Generate a random integer n between 10 and 100 inclusive. Generate n fair coin tosses and store them in a list, using "heads" and "tails" as the values. Assume that the sequence contains at least one heads. To guarantee this in your generated data, regenerate the sequence if it contains no heads; do not change individual tosses after generation. Use the selection algorithm to find the first heads. Display the list, then the toss number on the final line, optionally with a label. Tosses are numbered from 1, so the reported number is the index plus 1.

<details>
<summary><strong>Algorithm / pseudocode</strong> (click to expand)</summary>

```text
n ← random integer between 10 and 100
GENERATE n independent fair tosses INTO tosses
WHILE tosses contains no "heads":
    GENERATE a new sequence of n independent fair tosses INTO tosses
i ← 0
WHILE tosses[i] != "heads":
    i ← i + 1
DISPLAY tosses
DISPLAY a label and (i + 1)
```

</details>


In [ ]:
import random

n = random.randint(10, 100)
tosses = []
# Generate a sequence containing at least one heads.
# Apply the selection algorithm to find the first heads.


**Check your solution**

Run this cell immediately after running your solution.


In [ ]:
check("week06_ex04")


## Exercise 5 – Find Peter – linear search (10 minutes)

Ask for a positive integer n, then read n first names and store them in a list. Use linear search to find the first exact occurrence of "Peter". Print its zero-based index on the final line, optionally with a label. If Peter is absent, print -1. Stop searching at the first match. Names are case-sensitive: "peter" does not match "Peter".

<details>
<summary><strong>Algorithm / pseudocode</strong> (click to expand)</summary>

```text
READ n
names ← empty list
REPEAT n TIMES:
    READ name
    APPEND name TO names
i ← 0
WHILE i < LENGTH(names) AND names[i] != "Peter":
    i ← i + 1
IF i < LENGTH(names):
    index ← i
ELSE:
    index ← -1
DISPLAY a label and index
```

</details>


In [ ]:
n = int(input("Number of names: "))
names = []
# Read the names, then apply linear search.


**Check your solution**

Run this cell immediately after running your solution.


In [ ]:
check("week06_ex05")


## Exercise 6 – Largest random number – maximum selection (10 minutes)

Generate 15 random integers between 1000 and 2023 inclusive and store them in a list. Repeated numbers are allowed. Use the maximum selection algorithm to find the largest value, without max() or sorting. Display the list, then its largest value on the final line, optionally with a label. You may also display the maximum index before that final line.

<details>
<summary><strong>Algorithm / pseudocode</strong> (click to expand)</summary>

```text
numbers ← empty list
REPEAT 15 TIMES:
    APPEND random integer between 1000 and 2023 TO numbers
max_index ← 0
FOR i FROM 1 TO LENGTH(numbers) - 1:
    IF numbers[i] > numbers[max_index]:
        max_index ← i
DISPLAY numbers
DISPLAY a label and numbers[max_index]
```

</details>


In [ ]:
import random

numbers = []
# Generate 15 values and apply maximum selection.


**Check your solution**

Run this cell immediately after running your solution.


In [ ]:
check("week06_ex06")


## Exercise 7 – First perfect number in an interval (15 minutes)

A positive integer is perfect if it equals the sum of its positive divisors smaller than itself. For example, 6 = 1 + 2 + 3. Randomly generate an inclusive interval: choose the lower bound between 1 and 50, then choose the upper bound between that lower bound and 1000. Display both bounds together on one line (lower first, upper second). Find the first perfect number in the interval by checking candidates in increasing order. Stop the outer search as soon as one is found. Display the result on the final line, optionally with a label; if there is no match, display "There is no perfect number in this range". Use explicit divisor summation and a condition-controlled while search. This is Exercise 7 of the class exercise set.

<details>
<summary><strong>Algorithm / pseudocode</strong> (click to expand)</summary>

```text
GENERATE lower and upper
DISPLAY lower and upper on one line
candidate ← lower
found ← False
WHILE candidate <= upper AND NOT found:
    divisor_total ← 0
    FOR divisor FROM 1 TO candidate - 1:
        IF candidate MOD divisor = 0:
            divisor_total ← divisor_total + divisor
    IF divisor_total = candidate:
        found ← True
    ELSE:
        candidate ← candidate + 1
IF found:
    DISPLAY a label and candidate
ELSE:
    DISPLAY "There is no perfect number in this range"
```

</details>


In [ ]:
import random

lower = random.randint(1, 50)
upper = random.randint(lower, 1000)
print("Interval:", lower, upper)
# Search for the first perfect number in the interval.


**Check your solution**

Run this cell immediately after running your solution.


In [ ]:
check("week06_ex07")
